# Lọc + Chuẩn hoá Dataset, rồi Upload lên repo HF của bạn

> Chạy trên **Google Colab** (cache ephemeral, không dùng ổ local/Drive).

Dataset gốc `aisingapore/SEA-Instruct-2602` là **gated** nên không clone trực tiếp.
Notebook này sẽ:
1. Load dataset gốc bằng **streaming** (cần token đã được cấp quyền + 'write')
2. **Lọc rows** theo cột `prompt_primary_domain`
3. **Chỉ giữ 3 cột** (đúng thứ tự): `conversations_id`, `conversations`, `source` — xoá mọi cột khác
4. `conversations`: parse string → **list** (mỗi turn chỉ `content` + `role`)
5. Upload lên **repo HuggingFace của riêng bạn**

> Chỉ cần điền các phần `# >>> TODO` ở Cell cấu hình.

## 1. Cài đặt thư viện

In [ ]:
%pip install -q -U datasets huggingface_hub

## 2. Config

In [ ]:
# >>> TODO: Token HuggingFace (cần quyền truy cập dataset gated + quyền 'write' để upload)
HF_TOKEN = "hf_xxx_DIEN_TOKEN_CUA_BAN"

# Dataset gốc (gated) — chỉ load, không bị chỉnh sửa
SOURCE_REPO = "aisingapore/SEA-Instruct-2602"
SOURCE_CONFIG = "Vietnamese"

# >>> TODO: Repo đích trên account của bạn — sẽ được TẠO MỚI
TARGET_REPO = "tuongvyle/SEA-Instruct-2602-fine-tuned"
PRIVATE = True                  # True = private, False = public

# >>> TODO: Các domain muốn GIỮ LẠI (lọc theo cột 'prompt_primary_domain')
ALLOWED_DOMAINS = [
    'Social_and_Cultural_Issues',
    'Food_and_Cuisine',
    'Arts_and_Literature',
    'History_and_Heritage',
    'Sports_and_Fitness',
    'Daily_Life_and_Personal',
    'Government_and_Politics',
    'Parenting_and_Family',
    'Legal_Rights_and_Access',
    'Education',
    'Media_and_Entertainment',
    'Superstitions_Myth_and_Folklore',
    'Travel_and_Tourism',
    'Traditional_Medicine_and_Alternative_Healing',
    'Internet_and_Digital_Culture',
    'Agriculture_and_Fishing',
    'Religion_and_Belief',
    'Law_and_Justice',
    'Migrant_Worker_Expat_and_Student_Life'
]

# Cột dùng để LỌC rows
DOMAIN_COL = "prompt_primary_domain"

# 3 cột GIỮ LẠI (đúng thứ tự này), xoá tất cả cột còn lại
ID_COL   = "conversations_id"
CONV_COL = "conversations"
SRC_COL  = "source"
KEEP_COLS = [ID_COL, CONV_COL, SRC_COL]

## 3. Đăng nhập HuggingFace

In [ ]:
from huggingface_hub import login

login(token=HF_TOKEN)
print("Đăng nhập thành công.")

## 4. Tải dataset gốc (STREAMING — không ghi bản full xuống đĩa)

`streaming=True` đọc dần từng record qua mạng, **không** materialize toàn bộ dataset ra file Arrow → tránh lỗi hết dung lượng. Trả về `IterableDatasetDict`.

In [ ]:
from datasets import load_dataset

# streaming=True -> KHÔNG tải/ghi bản full xuống đĩa
stream = load_dataset(SOURCE_REPO, SOURCE_CONFIG, token=HF_TOKEN, streaming=True)

print(stream)
first_split = list(stream.keys())[0]

# Xem thử 1 record để biết tên cột
peek = next(iter(stream[first_split]))
print("\nCác cột:", list(peek.keys()))

In [ ]:
# (Tuỳ chọn) Lấy mẫu N record đầu để xem các giá trị có trong cột domain
# -> điền ALLOWED_DOMAINS cho đúng. Streaming nên không đếm toàn bộ (sẽ chậm).
from collections import Counter
from itertools import islice

SAMPLE_N = 5000
counts = Counter()
for row in islice(stream[first_split], SAMPLE_N):
    counts[row[DOMAIN_COL]] += 1

print(f"Các domain xuất hiện trong {SAMPLE_N} record đầu:")
for value, n in counts.most_common():
    print(f"{n:>8}  {value}")

## 5. Lọc + Chuẩn hoá + Upload theo shard (streaming, RAM thấp)

Một vòng duy nhất, **không gom hết vào RAM**:
- **Lọc** record theo `prompt_primary_domain` ∈ `ALLOWED_DOMAINS`
- **Chỉ giữ 3 cột** đúng thứ tự: `conversations_id`, `conversations`, `source`
- `conversations`: string → **list** (mỗi turn chỉ `content` + `role`)
- Cứ đủ `ROWS_PER_SHARD` record → ghi 1 shard parquet nhỏ → **upload** → xoá → giải phóng RAM

Có **resume** (bỏ qua shard đã có trên repo) + **retry** khi rớt mạng. Chạy được với data lớn bất kỳ, RAM/đĩa thấp.

In [ ]:
import json, ast, os, gc
from datasets import Dataset
from huggingface_hub import HfApi, create_repo

api = HfApi(token=HF_TOKEN)
create_repo(TARGET_REPO, repo_type="dataset", private=PRIVATE, exist_ok=True, token=HF_TOKEN)
existing = set(api.list_repo_files(TARGET_REPO, repo_type="dataset"))  # để RESUME

allowed_set = set(ALLOWED_DOMAINS)
KEEP_KEYS = ["content", "role"]
ROWS_PER_SHARD = 2000   # mỗi shard 2000 record -> RAM thấp, không gom hết vào bộ nhớ

TMP = "/content/hf_shards_tmp"   # đĩa Colab, mỗi lúc chỉ 1 shard rồi xoá
os.makedirs(TMP, exist_ok=True)

def parse_conv(conv):
    if isinstance(conv, str):
        try:
            return json.loads(conv)
        except Exception:
            return ast.literal_eval(conv)
    return conv

def build_row(row):
    conv = [{k: t.get(k) for k in KEEP_KEYS} for t in parse_conv(row[CONV_COL])]
    return {ID_COL: row[ID_COL], CONV_COL: conv, SRC_COL: row[SRC_COL]}

def flush(buf, split_name, idx):
    """Ghi 1 shard nhỏ ra đĩa rồi upload, KHÔNG giữ trong RAM."""
    path_in_repo = f"data/{split_name}-{idx:05d}.parquet"
    if path_in_repo in existing:
        print("skip (đã có):", path_in_repo); return
    local = os.path.join(TMP, f"{split_name}-{idx:05d}.parquet")
    Dataset.from_list(buf).to_parquet(local)
    for attempt in range(1, 6):
        try:
            api.upload_file(path_or_fileobj=local, path_in_repo=path_in_repo,
                            repo_id=TARGET_REPO, repo_type="dataset")
            print("OK:", path_in_repo, f"({len(buf)} records)"); break
        except Exception as e:
            print(f"  retry {attempt}/5: {e}")
    os.remove(local)

for split_name, split_stream in stream.items():
    buf, idx, seen, kept = [], 0, 0, 0
    for row in split_stream:
        seen += 1
        if row[DOMAIN_COL] in allowed_set:
            buf.append(build_row(row)); kept += 1
            if len(buf) >= ROWS_PER_SHARD:
                flush(buf, split_name, idx); idx += 1
                buf = []; gc.collect()        # giải phóng RAM
    if buf:
        flush(buf, split_name, idx)           # shard cuối
    print(f"==> {split_name}: duyệt {seen} -> giữ {kept} records")

print(f"\nXong -> https://huggingface.co/datasets/{TARGET_REPO}")

## 6. Kiểm tra lại dataset đã upload

Load thử vài record từ repo đích để xác nhận: đúng 3 cột và `conversations` là **list**.

In [ ]:
import json
from datasets import load_dataset

check = load_dataset(TARGET_REPO, split="train", streaming=True, token=HF_TOKEN)
ex = next(iter(check))

print("Các cột:", list(ex.keys()))   # phải là [conversations_id, conversations, source]
print("type(conversations):", type(ex[CONV_COL]).__name__, "| số lượt:", len(ex[CONV_COL]))
print("\nMẫu record:")
print("  conversations_id:", ex[ID_COL])
print("  source          :", ex[SRC_COL])
print("  conversations   :")
print(json.dumps(ex[CONV_COL], ensure_ascii=False, indent=2))